## Setting

Same primitives as v1/v2: two truncated bosonic modes $A,B$ plus a
dispersively coupled ancilla, with a variational sequence of the form

$$
\text{layer } k: \quad \mathrm{ECD}_A(\beta_A^{(k)}) \;\to\;
                         \mathrm{ECD}_B(\beta_B^{(k)}) \;\to\;
                         R(\theta^{(k)}, \varphi^{(k)}).
$$

Tensor ordering:
$\mathcal{H}_\text{qubit}\otimes\mathcal{H}_A\otimes\mathcal{H}_B$.

![ECD+R sequence](ecd_r_sequence.svg)

# Variational ECD+R decomposition of the two-mode squeezing gate

This notebook searches for the parameters of an **ECD + qubit-rotation (ECD+R)** circuit
that approximates the two-mode squeezing (TMS) unitary

$$
S_{AB}(r) \;=\; \exp\!\bigl[\, r\,(a_A^{\dagger} a_B^{\dagger} - a_A a_B)\,\bigr]
$$

acting on two bosonic modes $A$ and $B$ with the help of a single dispersively
coupled two-level ancilla ("qubit"). This is the standard hybrid CV–DV setting
used in circuit-QED experiments, where the qubit mediates non-Gaussian control
over otherwise-linear cavities.

The variational ansatz is built from three primitives:

1. **Echoed Conditional Displacement on mode $A$**,
   $\mathrm{ECD}_A(\beta) = |g\rangle\langle g|\otimes D_A(\beta/2)\otimes I_B
                          + |e\rangle\langle e|\otimes D_A(-\beta/2)\otimes I_B$.
2. **Echoed Conditional Displacement on mode $B$**,
   $\mathrm{ECD}_B(\beta) = |g\rangle\langle g|\otimes I_A\otimes D_B(\beta/2)
                          + |e\rangle\langle e|\otimes I_A\otimes D_B(-\beta/2)$.
3. A **qubit rotation** $R(\theta,\varphi) = \exp\!\bigl[-\tfrac{i\theta}{2}
   (\cos\varphi\,X + \sin\varphi\,Y)\bigr]$.

A layer consists of $\mathrm{ECD}_A \to \mathrm{ECD}_B \to R$, and the ansatz
stacks $N_{\text{layers}}$ of them. After the sequence, the qubit is
post-selected on $|g\rangle$; the remaining two-mode state is compared to the
target state $S_{AB}(r)|0,0\rangle$ (a two-mode squeezed vacuum) via
fidelity.


**Number of parameters per layer=6 (1 for each ECD and 2 for qubit rotation)**

# Variational ECD+R decomposition of the two-mode squeezing gate  —  v3

**Changes vs v2 (single-Kraus / probabilistic version):**

1. **Deterministic channel, post-selection not necessary.** v2 used only the top-left
   block $K = U_{gg} = (\langle g | \otimes I_2) U (|g\rangle \otimes I_2) $, i.e. the heralded Kraus operator
   for the outcome "ancilla measured in $|g\rangle$". v3 uses the full
   ancilla-traced channel

   $$
   \mathcal{E}(\rho) \;=\; U_{gg}\,\rho\,U_{gg}^{\dagger}
                        \,+\, U_{eg}\,\rho\,U_{eg}^{\dagger},
   $$

   whose Kraus operators are the two $d\times d$ blocks in the first block
   column of $U$. Unitarity of $U$ gives
   $U_{gg}^{\dagger}U_{gg} + U_{eg}^{\dagger}U_{eg} = I$, so $\mathcal{E}$ is
   CPTP — no post-selection, no dependence on ancilla-measurement outcomes.

2. **Cost is entanglement (process) fidelity of a channel to a unitary
   target.** For $\mathcal{E}$ with Kraus $\{K_i\}$ against target $V$,

   $$
   F_e(\mathcal{E},V) = \frac{1}{d^{2}}\sum_i |\mathrm{Tr}[V^{\dagger}K_i]|^{2}
   = \frac{|\mathrm{Tr}[V^{\dagger}U_{gg}]|^{2}
         + |\mathrm{Tr}[V^{\dagger}U_{eg}]|^{2}}{d^{2}}.
   $$

   $F_e = 1$ iff $\mathcal{E}(\rho)=V\rho V^\dagger$ exactly, i.e. iff one of
   $\{U_{gg}, U_{eg}\}$ equals $V$ (up to phase) and the other is zero. Both
   solutions are physically equivalent — they just differ in whether the
   ancilla deterministically ends in $|g\rangle$ or $|e\rangle$.

3. **No dependence on ancilla measurement probability.** In v2 the cost
   implicitly rewarded solutions with high $P_g$; v3 doesn't measure the
   ancilla at all. If a real experiment does measure and reset, either
   outcome is fine because the resulting cavity state is the same.

4. **Landscape symmetrized.** v2 only credited the $|g\rangle$-branch, so
   random initializations that happened to route amplitude through
   $|e\rangle$ were penalized. v3 credits both branches, which usually gives
   a smoother, more symmetric landscape.

In [1]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize
from functools import partial # for callback in minimization with additional argument

# circuit-blocks

In [2]:
def destroy(dim):
    a = np.zeros((dim, dim), dtype=complex)
    for n in range(1, dim):
        a[n - 1, n] = np.sqrt(n)
    return a


def displacement(alpha, mode):
    if mode == "A":
        return expm(alpha * adagA - np.conjugate(alpha) * aA)
    elif mode == "B":
        return expm(alpha * adagB - np.conjugate(alpha) * aB)
    raise ValueError("mode must be 'A' or 'B'")


def Rphi(theta, phi):
    sigma = np.cos(phi) * X + np.sin(phi) * Y
    return expm(-1j * theta * sigma / 2)


def ECD_A(beta):
    Dp = displacement( beta / 2, "A")
    Dm = displacement(-beta / 2, "A")
    return np.kron(Pg, np.kron(Dp, IB)) + np.kron(Pe, np.kron(Dm, IB))


def ECD_B(beta):
    Dp = displacement( beta / 2, "B")
    Dm = displacement(-beta / 2, "B")
    return np.kron(Pg, np.kron(IA, Dp)) + np.kron(Pe, np.kron(IA, Dm))


def embedded_rotation(theta, phi,d_cav):
    return np.kron(Rphi(theta, phi), np.eye(d_cav))




## Params

We got two types of qumode-Type A remains a qumode (so we keep high level Fock-truncation) **which one should we choose?**
Type B effectively works as a qubit (so we keep only 2-levels)

To represent Dicke model we need this 4 system qumode and 1 ancilla qubit. We can use the same ancilla qubit to entangle two qumodes (which in the Dicke model would imply entangling individual qubits with the cavity qumode). Note that for the open Dicke model, even though we need a ancilla qubit, we can reuse the same ancilla in principal.


In [3]:
cutoffA = 10  
cutoffB = 2
Nlayers_start = 1     # initial number of ECD+R layers
Nlayers_max   = 6     # layers are grown adaptively, capped at this value
F_target      = 0.999 # stop adding layers once process fidelity reaches this
r_target = 0.25

d_cav = cutoffA * cutoffB



aA, IA = destroy(cutoffA), np.eye(cutoffA)
adagA  = aA.conj().T
aB, IB = destroy(cutoffB), np.eye(cutoffB)
adagB  = aB.conj().T

X  = np.array([[0, 1],  [1, 0]],  dtype=complex)
Y  = np.array([[0, -1j],[1j, 0]], dtype=complex)
Pg = np.array([[1, 0], [0, 0]],   dtype=complex)
Pe = np.array([[0, 0], [0, 1]],   dtype=complex)

## Target unitary 

Here, we are doing two-mode squeezing (TMS) unitary

$$
S_{AB}(r) \;=\; \exp\!\bigl[\, r\,(a_A^{\dagger} a_B^{\dagger} - a_A a_B)\,\bigr]
$$

In [4]:

def two_mode_squeezing(r):
    a1 = np.kron(aA, IB)
    a2 = np.kron(IA, aB)
    ad1, ad2 = a1.conj().T, a2.conj().T
    G = ad1 @ ad2 - a1 @ a2
    return expm(r * G)

V_target = two_mode_squeezing(r_target)


# Cost function


Split the full $2d\times 2d$ ansatz unitary into four $d\times d$ blocks by
the qubit degree of freedom. With the qubit ordering $\{|g\rangle,|e\rangle\}$,

$$
U(\vec p) =
\begin{pmatrix} U_{gg} & U_{ge} \\ U_{eg} & U_{ee} \end{pmatrix}
=
\begin{pmatrix}
U[0{:}d,\,0{:}d]   & U[0{:}d,\,d{:}2d] \\
U[d{:}2d,\,0{:}d]  & U[d{:}2d,\,d{:}2d]
\end{pmatrix}.
$$

Initialize the qubit in $|g\rangle$ (the "0" of the qubit computational
basis, hence the "u_00 / u_10 blocks"). Acting on the joint state
$|g\rangle\otimes|\psi\rangle$,

$$
U|g,\psi\rangle
= |g\rangle\otimes U_{gg}|\psi\rangle
+ |e\rangle\otimes U_{eg}|\psi\rangle.
$$

Tracing out the qubit gives the deterministic cavity channel with two
Kraus operators $K_0 = U_{gg}$, $K_1 = U_{eg}$; unitarity of $U$ ensures
$K_0^{\dagger}K_0 + K_1^{\dagger}K_1 = I$. The entanglement fidelity to the
target unitary $V$ is then

$$
F_e(\vec p) = \frac{|\mathrm{Tr}[V^{\dagger} U_{gg}(\vec p)]|^{2}
                  + |\mathrm{Tr}[V^{\dagger} U_{eg}(\vec p)]|^{2}}{d^{2}}
                  \;\in\; [0,1].
$$


In [5]:
def ansatz(params, d_cav):
    n_layers = len(params) // 6  # number of layers grows adaptively, so infer it from params
    U = np.eye(2 * d_cav, dtype=complex) # 2 is the ancilla qubit dimension
    idx = 0
    for _ in range(n_layers):
        betaA = params[idx] + 1j * params[idx + 1]; idx += 2
        betaB = params[idx] + 1j * params[idx + 1]; idx += 2
        theta = params[idx]
        phi   = params[idx + 1]; idx += 2

        U = ECD_A(betaA)                  @ U
        U = ECD_B(betaB)                  @ U
        U = embedded_rotation(theta, phi,d_cav) @ U
    return U


def kraus_ops(params, d_cav):
    """Return the two Kraus operators (K0, K1) = (U_gg, U_eg) of the
    deterministic ancilla-traced cavity channel, starting from |g>.

    In the qubit ordering {|g>, |e>}, these are the two d x d blocks in the
    first block column of the full 2d x 2d ansatz unitary.
    """
    U = ansatz(params, d_cav)
    K0 = U[:d_cav,        :d_cav]   # U_gg  = <g| U |g> =U_00
    K1 = U[d_cav:2*d_cav, :d_cav]   # U_eg  = <e| U |g> =U_10
    return K0, K1


def process_fidelity(params, d_cav):
    """Entanglement fidelity F_e of the deterministic channel to V_target."""
    K0, K1 = kraus_ops(params, d_cav)
    Vd = V_target.conj().T
    tr0 = np.trace(Vd @ K0)
    tr1 = np.trace(Vd @ K1)
    return (np.abs(tr0)**2 + np.abs(tr1)**2) / (d_cav ** 2)


def cost(params, d_cav):
    return 1.0 - process_fidelity(params, d_cav)


# Diagnostic: which branch does the amplitude live in?  For a perfect
# solution one of these should be 1 and the other 0.  This tells us whether
# the qubit ends deterministically in |g> (P_g -> 1) or in |e> (P_g -> 0).
def branch_weights(params, d_cav):
    K0, K1 = kraus_ops(params, d_cav)
    w0 = np.real(np.trace(K0.conj().T @ K0)) / d_cav   # avg P_g
    w1 = np.real(np.trace(K1.conj().T @ K1)) / d_cav   # avg P_e
    return w0, w1


_history = []
def callback(xk, d_cav):
    _history.append(process_fidelity(xk, d_cav))
    if len(_history) % 10 == 0:
        print(f"Step {len(_history):4d}   F_e = {_history[-1]:.8f}")

##  Optimization

There is one **tuning parameter** that needs testing and better understanding which sets overall scale for the parameters of the angles.


In [6]:
tuning_factor = 0.3
rng = np.random.default_rng(seed=0)

Nlayers = Nlayers_start
params  = tuning_factor * rng.standard_normal(Nlayers * 6)

while True:
    _history.clear()
    result = minimize(
        cost, params, args=(d_cav,), method="BFGS", callback=partial(callback, d_cav=d_cav),
        options={"maxiter": 500, "disp": True},
    )
    params = result.x
    F_e = process_fidelity(params, d_cav)
    print(f"Nlayers = {Nlayers}   F_e = {F_e:.6f}")

    if F_e >= F_target or Nlayers >= Nlayers_max:
        break

    # target fidelity not reached yet: grow the ansatz by one more layer,
    # warm-started from the current solution plus a freshly initialized layer
    Nlayers += 1
    params = np.concatenate([params, tuning_factor * rng.standard_normal(6)])

w0, w1 = branch_weights(result.x, d_cav)
print()
print("Optimization finished")
print(f"Nlayers used          = {Nlayers}  (Nlayers_max = {Nlayers_max})")
print(f"Final F_e            = {process_fidelity(result.x, d_cav):.6f}")
print(f"Avg branch weight g  = {w0:.6f}   (want 1 or 0)")
print(f"Avg branch weight e  = {w1:.6f}   (want 0 or 1)")
print(f"Sum (should be 1)    = {w0+w1:.6f}")
print()

idx = 0
for k in range(Nlayers):
    betaA = result.x[idx] + 1j * result.x[idx + 1]; idx += 2
    betaB = result.x[idx] + 1j * result.x[idx + 1]; idx += 2
    theta = result.x[idx]
    phi   = result.x[idx + 1]; idx += 2
    print(f"Layer {k}")
    print(f"  betaA = {betaA}")
    print(f"  betaB = {betaB}")
    print(f"  theta = {theta}")
    print(f"  phi   = {phi}")
    print()

Optimization terminated successfully.
         Current function value: 0.253598
         Iterations: 5
         Function evaluations: 56
         Gradient evaluations: 8
Nlayers = 1   F_e = 0.746402
Step   10   F_e = 0.74640197
Optimization terminated successfully.
         Current function value: 0.253598
         Iterations: 11
         Function evaluations: 182
         Gradient evaluations: 14
Nlayers = 2   F_e = 0.746402
Step   10   F_e = 0.74635587
Step   20   F_e = 0.74639347
Optimization terminated successfully.
         Current function value: 0.253598
         Iterations: 27
         Function evaluations: 589
         Gradient evaluations: 31
Nlayers = 3   F_e = 0.746402
Step   10   F_e = 0.74620796
Step   20   F_e = 0.74639731
Step   30   F_e = 0.74640130
Step   40   F_e = 0.74640196
Optimization terminated successfully.
         Current function value: 0.253598
         Iterations: 42
         Function evaluations: 1125
         Gradient evaluations: 45
Nlayers = 4   F_e = 

## 7. Discussion


**Relation to v2.** v2's cost $F^{(v2)} = |\mathrm{Tr}[V^{\dagger} U_{gg}]|^2 / d^2$
is a strict lower bound on v3's cost:
$F_e = F^{(v2)} + |\mathrm{Tr}[V^{\dagger} U_{eg}]|^2 / d^2 \geq F^{(v2)}$.
The two optima coincide when the optimum solution routes all amplitude
through the $|g\rangle$ branch; v3 additionally accepts the mirror image
solution where all amplitude goes through the $|e\rangle$ branch. In
practice this makes optimization noticeably more robust to initialization
because random initial angles almost never concentrate on one branch.

**Diagnostic.** The two branch weights
$w_i = \mathrm{Tr}[K_i^{\dagger} K_i]/d$ sum to 1 by CPTP condition. When
$F_e \to 1$, one of them must go to 1 and the other to 0 — this is the
"deterministic disentanglement" condition. Watching $(w_0, w_1)$ during
optimization tells you which branch the optimizer picked; a stubborn
$(w_0, w_1) \approx (0.5, 0.5)$ with $F_e$ stuck below 1 is a signal that
`Nlayers` is too shallow to disentangle the ancilla.

**Reference.** The same channel-vs-unitary fidelity is standard in ECD-toolkit
compilations, where the ancilla is either reset via a final rotation or
absorbed as an environment — see Eickbusch et al., *Nat. Phys.* **18**, 1464
(2022), and the follow-up cascade / two-mode compilations, e.g. Diringer
et al., *arXiv:2308.16480*.
